In [3]:
import numpy as np
import matplotlib.pyplot as plt
import time
from scipy.spatial import cKDTree
from tqdm.notebook import tqdm

class Detector:
    def __init__(self, R=15.0, N=2000, r=0.1, a=0.5):
        self.R = R
        self.N = N
        self.r = r
        self.a = a
        self.pmt_positions = self._generate_pmt_fibonacci()
        self.kdtree = cKDTree(self.pmt_positions)        # ОПТИМИЗАЦИЯ KD-дерево 

    def _generate_pmt_fibonacci(self):
        indices = np.arange(0, self.N, dtype=float) + 0.5
        phi = np.arccos(1 - 2 * indices / self.N)
        theta = np.pi * (1 + 5**0.5) * indices
        x = np.cos(theta) * np.sin(phi) * self.R
        y = np.sin(theta) * np.sin(phi) * self.R
        z = np.cos(phi) * self.R
        return np.column_stack([x, y, z])

    def simulate_event_slow(self, E, Y=10000):
        N_photons = int(Y * E)
        directions = np.random.normal(size=(N_photons, 3))
        directions /= np.linalg.norm(directions, axis=1, keepdims=True)
        hit_points = directions * self.R
        
        hits = np.zeros(self.N, dtype=int)
        for p in hit_points:
            dists = np.linalg.norm(self.pmt_positions - p, axis=1)
            idx = np.where(dists <= self.r)[0]
            hits[idx] += 1
            
        return np.sum(np.random.poisson(self.a * hits))

    def simulate_event_fast(self, E, Y=10000):
        N_photons = int(Y * E)
        directions = np.random.normal(size=(N_photons, 3))
        directions /= np.linalg.norm(directions, axis=1, keepdims=True)
        hit_points = directions * self.R
        nearby_pmts_list = self.kdtree.query_ball_point(hit_points, self.r)
        hits = np.zeros(self.N, dtype=int)
        for idx_list in nearby_pmts_list:
            if idx_list: 
                hits[idx_list] += 1
                
        return np.sum(np.random.poisson(self.a * hits))

np.random.seed(4)
detector = Detector(R=15.0, N=2000, r=0.1, a=0.5)



N_events_to_test = 50
start = time.time()
for _ in range(N_events_to_test):
    detector.simulate_event_slow(E=1.0)
time_slow = time.time() - start


start = time.time()
for _ in range(N_events_to_test):
    detector.simulate_event_fast(E=1.0)
time_fast = time.time() - start

print("\n--- РЕЗУЛЬТАТЫ ---")
print(f"Время медленного алгоритма: {time_slow:.2f} сек.")
print(f"Время быстрого алгоритма:   {time_fast:.2f} сек.")
print(f"Ускорение: {time_slow / time_fast:.1f} раз!")

SyntaxError: invalid syntax (1876513861.py, line 59)